# Part 3 - Investment Analysis Agentic System (Agno, formerly phidata)

*Last updated: November 2026*

You will build a team of specialist agents, each with its own role, instructions and tools:
- **Market Sentiment Evaluator** - searches the web and Yahoo Finance news and scores sentiment.
- **Financial Insights Agent** - stock prices, analyst ratings and key metrics.
- **Deep Financial Analytics Agent** - ratios, income statements and price history.
- **Strategic Investment Advisor** - turns the findings into a BUY / SELL / HOLD view.

A **team leader** coordinates the specialists and combines their results into one report.
First, a single agent with a **calculator tool** shows how agents call tools step by step.

> Educational example only - not financial advice.

## How to run this notebook

1. Set up the course environment once, in the course folder: `uv sync`.
   New to coding? Follow `BEGINNER_SETUP.md` step by step. Otherwise see `LOCAL_SETUP.MD`.
2. Select the **`.venv`** Python kernel (*Python 3.12*) for this notebook.
3. Copy `.env.example` (course folder) to `.env` and add your keys. The notebook finds `.env` automatically.

**Keys and settings used in this notebook**
- `OPENAI_API_KEY` (required)

Run the cells from top to bottom.

> **Compatibility update (differs from the recording)**
> - The `phidata` library has been renamed **Agno**. Imports change from `phi.*` to `agno.*`; the agents, tools and instructions are the same.
> - `Calculator` → `CalculatorTools` (same options).
> - **News search changed:** the recording used `GoogleSearch` and Yahoo Finance news. Google now blocks the simple search scraper
>   (it returns no results) and the Yahoo Finance news feed was failing in November 2026, so the Market Sentiment Evaluator now uses
>   **`DuckDuckGoTools`** (web and news search, no API key). The stock-price and financial-data tools still use Yahoo Finance.
> - The team is now built with Agno's dedicated `Team` class (`members=[...]`) instead of `Agent(team=[...])`.
> - The final prompt analyses the most recent month instead of the fixed date "January 2025", so it matches the live market data.
> - The unused `Groq` import was removed.

In [ ]:
# Setup check: run this cell first.
# Nothing is installed here. All packages were installed once with `uv sync` (see LOCAL_SETUP.MD).
import sys
from pathlib import Path

python_version = sys.version.split()[0]

if sys.version_info[:2] != (3, 12):
    print(f"Warning: this course is tested on Python 3.12, but this kernel runs Python {python_version}.")
    print("Select the .venv kernel - see 'How to run this notebook' above.")
elif Path(sys.prefix).name != ".venv":
    print("Warning: this kernel is not the course .venv environment, so some packages may be missing.")
    print("Select the .venv kernel - see 'How to run this notebook' above.")
else:
    print("Ready: course .venv kernel | Python", python_version)

In [ ]:
# Load API keys from the .env file, so keys are never written in the notebook.
# load_dotenv() looks for .env in this folder and then in each parent folder (the course folder).
import os

from dotenv import load_dotenv

load_dotenv()

REQUIRED_KEYS = ['OPENAI_API_KEY']

missing = [key for key in REQUIRED_KEYS if not os.getenv(key)]
if missing:
    print("Missing required keys:", ", ".join(missing), "- see 'How to run this notebook' above.")
else:
    print("All required API keys are loaded.")

In [ ]:
# Hide library deprecation warnings so the lesson output stays readable
import warnings
warnings.filterwarnings("ignore")

In [ ]:
## Libraries ("phi" is now "agno")
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.team.team import Team
from agno.tools.calculator import CalculatorTools
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.yfinance import YFinanceTools

## 1. A single agent with a calculator tool

In [ ]:
agent = Agent(
    tools=[
        CalculatorTools(
            add=True,
            subtract=True,
            multiply=True,
            divide=True,
            exponentiate=True,
            factorial=True,
            is_prime=True,
            square_root=True,
        )
    ],
    show_tool_calls=True,
    markdown=True,
)
agent.print_response("What is 10*5 then to the power of 2, do it step by step")

## 2. Specialist agents

In [ ]:
# Market Sentiment Evaluator
market_sentiment_bot = Agent(
    name="Market Sentiment Evaluator",
    role="Analyze and interpret financial news articles",
    model=OpenAIChat(id="gpt-4o"),
    tools=[DuckDuckGoTools()],  # web search + news search (replaces GoogleSearch and Yahoo news)
    instructions=[
        "Identify relevant news articles for each company and assess their sentiment critically.",
        "Assign sentiment ratings from 1 (very negative) to 10 (very positive) with justification and references.",
        "Provide specific sources, detailed reasoning, and relevant article links."
    ],
    show_tool_calls=True,
    markdown=True,
)

In [ ]:
# Financial Insights Agent
financial_data_bot = Agent(
    name="Financial Insights Agent",
    role="Extract and analyze essential company financial data with a structured analytical approach",
    model=OpenAIChat(id="gpt-4o"),
    tools=[YFinanceTools(stock_price=True, company_info=True, stock_fundamentals=True, analyst_recommendations=True)],
    instructions=[
        "Gather stock prices, analyst ratings, and key financial metrics.",
        "Emphasize company fundamentals and trends, structuring the data in tables with meaningful insights."
    ],
    show_tool_calls=True,
    markdown=True,
)

In [ ]:
# Deep Financial Analytics Agent
deep_financials_bot = Agent(
    name="Deep Financial Analytics Agent",
    role="Extract and analyze extensive company financial data with a structured and data-driven approach",
    model=OpenAIChat(id="gpt-4o"),
    tools=[YFinanceTools(company_info=True, key_financial_ratios=True, income_statements=True, historical_prices=True)],
    instructions=[
        "Gather key financial ratios, income statements, balance sheets, cash flow reports, and historical price data.",
        "Highlight company fundamentals and trends, structuring the data in tables with meaningful insights."
    ],
    show_tool_calls=True,
    markdown=True,
)

In [ ]:
# Strategic Investment Advisor
investment_strategy_bot = Agent(
    name="Strategic Investment Advisor",
    role="An experienced portfolio manager tasked with making data-driven investment decisions through critical evaluation.",
    model=OpenAIChat(id="gpt-4o"),
    instructions=[
        "Verify outputs for accuracy and completeness.",
        "Analyze data holistically to determine a final sentiment score (1-10) with justification.",
        "After careful assessment, provide a clear BUY, SELL, or HOLD recommendation for the stock."
    ],
    show_tool_calls=True,
    markdown=True,
)

## 3. The investment team

In [ ]:
# Unified Investment Intelligence Team
from datetime import date

today = date.today().strftime("%Y-%m-%d")

investment_analysis_team = Team(
    model=OpenAIChat(id="gpt-4o"),
    members=[market_sentiment_bot, financial_data_bot, deep_financials_bot, investment_strategy_bot],
    mode="coordinate",
    instructions=[
        "Integrate insights from all specialized agents to deliver a comprehensive, data-backed investment analysis.",
        f"Ensure all references and data points include timestamps, maintaining relevance as of {today}.",
        "Structure findings in clear tables for transparency, ensuring conclusions are data-driven.",
        "Explain the analytical methodology and always provide a definitive BUY, SELL, or HOLD recommendation."
    ],
    show_tool_calls=True,
    markdown=True,
)

In [ ]:
investment_analysis_team.print_response(
    f"Analyze the sentiment for the following companies for the last month (today is {today}): AMD, TSM. \n\n"
    "1. **Sentiment Analysis**: Search for relevant news articles and interpret the sentiment for each company. Provide the sentiment score"
    "2. **Financial Data**: Analyze stock price and company fundamental financial metrics and other resources. Highlight key trends"
    "3. **Consolidated Analysis**: Combine the insights from the various srouces such as sentiment analysis and financial data to assign a final BUY, SELL or HOLD recommendation"
    "Ensure your response is accurate, comprehensive, analytical and well-reasoned. Include references to sources with publication dates.",
    stream=True
)